# RE:GEN AI — Multi-Agent Sustainability Command Center

**Google Kaggle AI Agents: Intensive Vibe Coding Capstone Project 2026**

This notebook demonstrates the core agent pipeline of RE:GEN AI using self-contained sample data.
No backend server is required to run this notebook.

> **Disclaimer:** RE:GEN AI is a prototype decision-support system. All data shown is simulated
> for capstone demonstration and does not represent real sensor readings. This is not professional
> regulatory, financial, or engineering advice.

## 1. Setup

Install required packages. Gemini is optional — the notebook runs fully without it.

In [ ]:
# Install dependencies (Kaggle environment usually has pandas already)
import subprocess
import sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

try:
    import pandas as pd
except ImportError:
    install('pandas')
    import pandas as pd

try:
    from google import genai
    GENAI_AVAILABLE = True
except ImportError:
    try:
        install('google-genai')
        from google import genai
        GENAI_AVAILABLE = True
    except Exception:
        GENAI_AVAILABLE = False

import os, json, math
from datetime import date, timedelta
import random

print('pandas:', pd.__version__)
print('google-genai available:', GENAI_AVAILABLE)

## 2. Simulated Campus Data

RE:GEN AI uses simulated smart-campus resource logs. In the full application these are loaded from
CSV files in `backend/data/`. Here we generate equivalent sample data inline so the notebook is
fully self-contained and reproducible on Kaggle.

In [ ]:
random.seed(42)

# --- 7-day water usage data (hourly, 2 locations) ---
water_rows = []
locations = ['Block-B Hostel', 'Lab Block']
base_date = date(2025, 6, 28)

for day_offset in range(7):
    d = (base_date + timedelta(days=day_offset)).isoformat()
    for hour in range(24):
        for loc in locations:
            if hour in range(0, 6):
                base = 12
                # inject anomaly on day 2-3 at night for Lab Block
                is_anomaly = (loc == 'Lab Block' and day_offset in (1, 2) and hour in (1, 2, 3))
            else:
                base = 80 + random.randint(-10, 10)
                is_anomaly = False
            usage = (base + random.randint(40, 80)) if is_anomaly else (base + random.randint(-3, 5))
            water_rows.append({'date': d, 'hour': hour, 'location': loc,
                                'usage_liters': max(0, usage), 'anomaly': is_anomaly})

water_df = pd.DataFrame(water_rows)
print(f'Water data: {len(water_df)} records, {water_df["anomaly"].sum()} anomalous readings')
water_df[water_df['anomaly']].head(6)

In [ ]:
# --- 7-day energy usage data (hourly, 3 zones) ---
energy_rows = []
zones = ['Seminar Hall', 'Computer Lab', 'Admin Block']
AFTER_HOURS = list(range(0, 6)) + [22, 23]

for day_offset in range(7):
    d = (base_date + timedelta(days=day_offset)).isoformat()
    for hour in range(24):
        for zone in zones:
            equipment = 'AC' if zone == 'Seminar Hall' else ('PC' if zone == 'Computer Lab' else 'Lighting')
            if hour in AFTER_HOURS:
                base = 2.5
                is_anomaly = (zone == 'Seminar Hall' and day_offset in (3, 4) and hour in (22, 23, 0, 1))
            else:
                base = 25 + random.uniform(-3, 3)
                is_anomaly = False
            usage = (base + random.uniform(15, 35)) if is_anomaly else (base + random.uniform(-0.5, 1.5))
            energy_rows.append({'date': d, 'hour': hour, 'zone': zone,
                                 'usage_kwh': round(max(0, usage), 2),
                                 'equipment': equipment, 'anomaly': is_anomaly})

energy_df = pd.DataFrame(energy_rows)
print(f'Energy data: {len(energy_df)} records, {energy_df["anomaly"].sum()} anomalous readings')
energy_df[energy_df['anomaly']].head(6)

## 3. Water Leakage Agent

Detects night-flow anomalies (0–5 AM), computes wasted liters vs. normal baseline,
assigns severity, and returns estimated cost and CO2 impact.

In [ ]:
def analyze_water(df):
    NIGHT_HOURS = list(range(0, 6))
    WATER_COST_PER_LITER = 0.05

    anomaly_rows = df[df['anomaly'] == True].copy()
    normal_rows  = df[df['anomaly'] == False].copy()
    night_normal = normal_rows[normal_rows['hour'].isin(NIGHT_HOURS)]
    baseline     = night_normal['usage_liters'].mean() if len(night_normal) > 0 else 12

    total_anomaly_liters  = float(anomaly_rows['usage_liters'].sum())
    expected_for_anomaly  = baseline * len(anomaly_rows)
    wasted_liters         = round(total_anomaly_liters - expected_for_anomaly, 1)

    if   wasted_liters > 1000: severity, severity_score = 'critical', 10
    elif wasted_liters > 500:  severity, severity_score = 'high',     30
    elif wasted_liters > 200:  severity, severity_score = 'medium',   55
    elif wasted_liters > 50:   severity, severity_score = 'low',      75
    else:                      severity, severity_score = 'none',     95

    return {
        'agent': 'Water Leakage Agent',
        'total_anomaly_readings': len(anomaly_rows),
        'baseline_night_usage_liters_per_hour': round(baseline, 1),
        'total_wasted_liters': wasted_liters,
        'severity': severity,
        'severity_score': severity_score,
        'estimated_cost_inr': round(wasted_liters * WATER_COST_PER_LITER, 2),
        'co2_equivalent_kg': round(wasted_liters * 0.001, 2),
        'data_notice': 'Simulated smart-campus resource logs — capstone demonstration only.',
    }

water_result = analyze_water(water_df)
print('=== Water Leakage Agent ===')
for k, v in water_result.items():
    print(f'  {k}: {v}')

## 4. Energy Optimization Agent

Identifies after-hours energy waste (10 PM–6 AM), computes kWh lost, cost impact at ₹8/kWh,
and CO2 equivalent using India grid emission factor (0.82 kg CO2/kWh).

In [ ]:
def analyze_energy(df):
    AFTER_HOURS = list(range(0, 6)) + [22, 23]
    ELECTRICITY_COST_PER_KWH = 8.0

    anomaly_rows = df[df['anomaly'] == True].copy()
    normal_after = df[(df['anomaly'] == False) & (df['hour'].isin(AFTER_HOURS))]
    baseline     = normal_after['usage_kwh'].mean() if len(normal_after) > 0 else 2.5

    total_anomaly_kwh = float(anomaly_rows['usage_kwh'].sum())
    expected_kwh      = baseline * len(anomaly_rows)
    wasted_kwh        = round(total_anomaly_kwh - expected_kwh, 2)

    if   wasted_kwh > 200: severity, severity_score = 'critical', 10
    elif wasted_kwh > 100: severity, severity_score = 'high',     28
    elif wasted_kwh > 50:  severity, severity_score = 'medium',   52
    elif wasted_kwh > 10:  severity, severity_score = 'low',      72
    else:                  severity, severity_score = 'none',     92

    return {
        'agent': 'Energy Optimization Agent',
        'total_anomaly_readings': len(anomaly_rows),
        'baseline_after_hours_kwh_per_hour': round(baseline, 2),
        'total_wasted_kwh': wasted_kwh,
        'severity': severity,
        'severity_score': severity_score,
        'estimated_cost_inr': round(wasted_kwh * ELECTRICITY_COST_PER_KWH, 2),
        'co2_equivalent_kg': round(wasted_kwh * 0.82, 2),
        'data_notice': 'Simulated smart-campus resource logs — capstone demonstration only.',
    }

energy_result = analyze_energy(energy_df)
print('=== Energy Optimization Agent ===')
for k, v in energy_result.items():
    print(f'  {k}: {v}')

## 5. Waste-to-Wealth Agent

Looks up a waste material in a knowledge base, applies hazard guardrails,
and returns a recovery pathway with an estimated value range.

> Exact profit is never claimed. All financial figures are estimates. Hazardous materials
> suppress financial data entirely and show a regulatory warning.

In [ ]:
# Inline knowledge base (subset of the full 30-material KB used in the app)
# Hazard levels match the production backend/data/waste_knowledge_base.json:
#   'high' and 'critical' trigger financial suppression; 'medium' and below do not.
WASTE_KB = {
    'e-waste': {
        'category': 'Electronics',
        'hazard_level': 'high',
        'estimated_value_range': {'min': 15, 'max': 45, 'unit': 'INR/kg'},
        'possible_products': ['refurbished components', 'extracted metals', 'circuit board materials'],
        'recommended_pathway': 'certified e-waste recycler',
        'buyer_types': ['authorized e-waste recyclers', 'government e-waste collection centers'],
        'hidden_value_score': 78,
        'sustainability_notes': 'Contains recoverable copper, gold, and silver traces.',
        'risks': 'Contains lead and cadmium — must use certified recycler.',
    },
    'paper': {
        'category': 'Paper & Cardboard',
        'hazard_level': 'none',
        'estimated_value_range': {'min': 8, 'max': 14, 'unit': 'INR/kg'},
        'possible_products': ['recycled paper', 'cardboard', 'packaging material'],
        'recommended_pathway': 'paper recycler / kabadiwala',
        'buyer_types': ['paper mills', 'local recyclers'],
        'hidden_value_score': 60,
        'sustainability_notes': 'Reduces deforestation pressure.',
        'risks': 'Low risk. Contaminated paper (food-soiled) has lower value.',
    },
    'chemical waste': {
        'category': 'Hazardous Chemical',
        'hazard_level': 'critical',
        'estimated_value_range': {'min': 0, 'max': 0, 'unit': 'INR/kg'},
        'possible_products': [],
        'recommended_pathway': 'licensed hazardous waste handler',
        'buyer_types': [],
        'hidden_value_score': 5,
        'sustainability_notes': 'Must comply with Hazardous Waste Management Rules 2016.',
        'risks': 'Serious environmental and health hazard. Do not dispose in general waste.',
    },
}

# Matches backend/core/guardrails.py: only 'high' and 'critical' suppress financial figures.
# 'medium' materials show estimated recovery values with appropriate qualifiers.
HAZARD_SUPPRESS_LEVELS = {'high', 'critical'}

def analyze_waste(waste_type: str, quantity_kg: float):
    key      = waste_type.lower().strip()
    material = WASTE_KB.get(key)

    if quantity_kg <= 0 or quantity_kg > 100000:
        return {'error': 'Quantity must be between 0 and 100,000 kg.'}

    if material is None:
        close = [k for k in WASTE_KB if key in k or k in key]
        return {'status': 'unknown_material', 'waste_type': waste_type,
                'message': f"'{waste_type}' not in knowledge base.", 'suggestions': close[:3]}

    hazard   = material['hazard_level']
    suppress = hazard in HAZARD_SUPPRESS_LEVELS

    if suppress:
        estimated_recovery = None
        hazard_warning     = True
        hazard_message     = (f'WARNING: {waste_type} is classified as {hazard} hazard. '
                               'Financial recovery data is suppressed. '
                               'Engage a licensed hazardous waste handler immediately.')
    else:
        mn = material['estimated_value_range']['min']
        mx = material['estimated_value_range']['max']
        estimated_recovery = {
            'min_inr': round(mn * quantity_kg, 2),
            'max_inr': round(mx * quantity_kg, 2),
            'unit_rate': f"{mn}-{mx} INR/kg",
            'note': 'Estimated only. Actual market prices vary.',
        }
        hazard_warning = False
        hazard_message = None

    return {
        'agent': 'Waste-to-Wealth Agent',
        'status': 'analyzed',
        'waste_type': waste_type,
        'quantity_kg': quantity_kg,
        'category': material['category'],
        'hazard_level': hazard,
        'hazard_warning': hazard_warning,
        'hazard_message': hazard_message,
        'possible_products': material['possible_products'],
        'recommended_pathway': material['recommended_pathway'],
        'hidden_value_score': material['hidden_value_score'],
        'estimated_recovery': estimated_recovery,
        'risks': material['risks'],
        'disclaimer': 'RE:GEN AI is a prototype. Not professional advice.',
    }

# Test 1: e-waste (high hazard — financial data suppressed, matches backend behavior)
print('=== Waste Agent: e-waste 50kg ===')
r1 = analyze_waste('e-waste', 50)
for k, v in r1.items():
    print(f'  {k}: {v}')

print()
# Test 2: paper (non-hazardous — shows full recovery pathway and estimated value)
print('=== Waste Agent: paper 100kg ===')
r2 = analyze_waste('paper', 100)
for k, v in r2.items():
    print(f'  {k}: {v}')

print()
# Test 3: chemical waste (critical hazard — all financials suppressed)
print('=== Waste Agent: chemical waste 20kg (hazard guardrail) ===')
r3 = analyze_waste('chemical waste', 20)
for k, v in r3.items():
    print(f'  {k}: {v}')

## 6. Pollution & Impact Agent

Aggregates water and energy savings into total CO2 reduction and
expresses impact in relatable equivalents.

In [ ]:
def analyze_impact(water_saved_liters, energy_saved_kwh, waste_value_inr=0):
    water_co2  = round(water_saved_liters * 0.001, 2)
    energy_co2 = round(energy_saved_kwh   * 0.82,  2)
    total_co2  = round(water_co2 + energy_co2, 2)

    trees_equiv    = round(total_co2 / 100 * 4.5, 1)
    vehicle_km     = round(total_co2 / 0.167, 1)
    household_days = round(total_co2 / 2.87, 1)

    annual = {
        'water_liters':  round(water_saved_liters * 52),
        'energy_kwh':    round(energy_saved_kwh * 52, 1),
        'co2_kg':        round(total_co2 * 52, 1),
        'financial_inr': round((water_saved_liters * 0.05 + energy_saved_kwh * 8.0) * 52),
        'note': 'Simulated projection — assumes same weekly loss rate sustained for 52 weeks.',
    }

    if   total_co2 >= 500: rating, score = 'Outstanding', 88
    elif total_co2 >= 200: rating, score = 'Excellent',   75
    elif total_co2 >= 100: rating, score = 'Good',        62
    elif total_co2 >= 30:  rating, score = 'Moderate',    48
    else:                  rating, score = 'Developing',  35

    return {
        'agent': 'Pollution & Impact Agent',
        'water_co2_saved_kg': water_co2,
        'energy_co2_saved_kg': energy_co2,
        'total_co2_saved_kg': total_co2,
        'trees_equivalent': trees_equiv,
        'vehicle_km_equivalent': vehicle_km,
        'household_days_equivalent': household_days,
        'sustainability_rating': rating,
        'sustainability_score': score,
        'annual_projections': annual,
    }

impact_result = analyze_impact(
    water_result['total_wasted_liters'],
    energy_result['total_wasted_kwh'],
)
print('=== Pollution & Impact Agent ===')
for k, v in impact_result.items():
    print(f'  {k}: {v}')

## 7. Decision Engine

Scores and ranks interventions by a weighted composite:
urgency (35%) + cost saving (30%) + environmental impact (25%) + feasibility (10%).

In [ ]:
def _score_action(urgency, cost_saving, env_impact, feasibility):
    return round(
        urgency             * 0.35
        + min(cost_saving / 1000, 30) * 0.30
        + env_impact        * 0.25
        + feasibility       * 0.10,
        2,
    )

def generate_decisions(water_r, energy_r):
    urgency_map = {'critical': 10, 'high': 8, 'medium': 5, 'low': 3, 'none': 1}
    actions = []

    wu = urgency_map.get(water_r['severity'], 1)
    actions.append({
        'id': 'W1', 'domain': 'Water',
        'issue': f"Night-time leakage ({water_r['total_wasted_liters']} L wasted)",
        'urgency': wu, 'urgency_label': water_r['severity'].upper(),
        'cost_saving_inr': water_r['estimated_cost_inr'],
        'env_impact_score': min(water_r['total_wasted_liters'] / 100, 10),
        'feasibility': 9,
        'priority_score': _score_action(wu, water_r['estimated_cost_inr'],
                                         min(water_r['total_wasted_liters'] / 100, 10), 9),
        'recommended_action': 'Inspect night-flow pipes at anomalous locations.',
        'timeline': 'Immediate' if water_r['severity'] in ('critical', 'high') else 'Within 7 days',
    })

    eu = urgency_map.get(energy_r['severity'], 1)
    actions.append({
        'id': 'E1', 'domain': 'Energy',
        'issue': f"After-hours waste ({energy_r['total_wasted_kwh']} kWh wasted)",
        'urgency': eu, 'urgency_label': energy_r['severity'].upper(),
        'cost_saving_inr': energy_r['estimated_cost_inr'],
        'env_impact_score': min(energy_r['total_wasted_kwh'] / 20, 10),
        'feasibility': 8,
        'priority_score': _score_action(eu, energy_r['estimated_cost_inr'],
                                         min(energy_r['total_wasted_kwh'] / 20, 10), 8),
        'recommended_action': 'Shutdown non-essential AC and lighting in flagged zones.',
        'timeline': 'Immediate' if energy_r['severity'] in ('critical', 'high') else 'Within 7 days',
    })

    actions.sort(key=lambda x: x['priority_score'], reverse=True)
    for i, a in enumerate(actions): a['rank'] = i + 1

    return {
        'agent': 'Decision Engine Agent',
        'ranked_actions': actions,
        'total_actions': len(actions),
        'total_potential_saving_inr': round(sum(a['cost_saving_inr'] for a in actions), 2),
        'top_priority_domain': actions[0]['domain'] if actions else None,
    }

decision_result = generate_decisions(water_result, energy_result)
print('=== Decision Engine ===')
print(f"  Top priority: {decision_result['top_priority_domain']}")
print(f"  Total estimated potential saving: ₹{decision_result['total_potential_saving_inr']}")
print()
for a in decision_result['ranked_actions']:
    print(f"  Rank {a['rank']}: [{a['domain']}] {a['issue']}")
    print(f"    Priority score: {a['priority_score']} | Urgency: {a['urgency_label']} | Timeline: {a['timeline']}")
    print(f"    Action: {a['recommended_action']}")
    print()

## 8. RE:GEN Score Agent

Produces a campus sustainability health index (0–100) with a before/after projection.

In [ ]:
def compute_regen_score(water_r, energy_r, impact_r, decision_r):
    wasted_liters = water_r['total_wasted_liters']
    wasted_kwh    = energy_r['total_wasted_kwh']
    co2_score     = impact_r['sustainability_score']

    water_pot  = min(100, 100 - (wasted_liters / 10))
    energy_pot = min(100, 100 - (wasted_kwh    / 5))
    waste_rec  = 60
    feasibility = 82

    urgency_avg = sum(a['urgency'] * 10 for a in decision_r['ranked_actions']) / max(len(decision_r['ranked_actions']), 1)
    urgency_red = max(0, min(100, 100 - urgency_avg))

    def weighted(wr, wa, en, co2, ur, fe):
        return round(wr * 0.20 + wa * 0.20 + en * 0.20 + co2 * 0.20 + ur * 0.10 + fe * 0.10, 1)

    before = weighted(
        max(0, waste_rec   - 25), max(0, water_pot  - 20),
        max(0, energy_pot  - 20), max(0, co2_score  - 15),
        max(0, urgency_red - 15), feasibility - 10,
    )
    after = weighted(waste_rec, water_pot, energy_pot, co2_score, urgency_red, feasibility)

    def grade(s):
        if s >= 90: return 'A+'
        if s >= 80: return 'A'
        if s >= 70: return 'B+'
        if s >= 60: return 'B'
        if s >= 50: return 'C+'
        if s >= 40: return 'C'
        if s >= 30: return 'D'
        return 'F'

    return {
        'agent': 'RE:GEN Score Agent',
        'before_score': before,
        'after_score': after,
        'improvement': round(after - before, 1),
        'current_grade': grade(before),
        'target_grade': grade(after),
        'interpretation': (
            f'Current campus health is Grade {grade(before)} ({before}/100). '
            f'Implementing all recommendations is projected to raise it to '
            f'Grade {grade(after)} ({after}/100) — a +{round(after - before, 1)}-point improvement.'
        ),
    }

score_result = compute_regen_score(water_result, energy_result, impact_result, decision_result)
print('=== RE:GEN Score Agent ===')
for k, v in score_result.items():
    print(f'  {k}: {v}')

## 9. Gemini Layer (Optional)

If a Gemini API key is available, this section calls Gemini 2.5 Flash Lite to generate
a plain-language explanation of the top-priority intervention.

If no key is available, a deterministic fallback is used — all agent outputs above are
fully computed without Gemini.

In [ ]:
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')

top_action = decision_result['ranked_actions'][0]

if GENAI_AVAILABLE and GEMINI_API_KEY:
    try:
        client = genai.Client(api_key=GEMINI_API_KEY)
        prompt = f"""You are a campus sustainability decision analyst. In exactly 2 clear sentences,
explain to a university sustainability officer WHY this action must be done FIRST.

Action: {top_action['recommended_action']}
Domain: {top_action['domain']}
Urgency level: {top_action['urgency_label']} ({top_action['urgency']}/10)
Estimated saving: Rs. {top_action['cost_saving_inr']:.0f}
Priority score: {top_action['priority_score']}

Rules:
- Say 'estimated' for all financial values
- Reference the specific numbers above
- Tell the officer exactly what to do within 24 hours
- Do not use: revolutionary, powerful AI, real-time intelligence, next-generation"""

        response = client.models.generate_content(
            model='gemini-2.5-flash-lite',
            contents=prompt,
        )
        gemini_text = response.text.strip()
        print('=== Gemini 2.5 Flash Lite — Decision Explanation ===')
        print(gemini_text)
        print('  [Gemini-powered]')
    except Exception as e:
        print(f'Gemini call failed ({e}). Using deterministic fallback.')
        gemini_text = None
else:
    gemini_text = None
    print('No GEMINI_API_KEY found — using deterministic fallback.')

if not gemini_text:
    fallback = (
        f"Prioritise '{top_action['recommended_action']}' (score {top_action['priority_score']}) "
        f"because urgency is {top_action['urgency']}/10 with an estimated weekly saving of "
        f"Rs. {top_action['cost_saving_inr']:.0f}. "
        f"Dispatch the maintenance team within 24 hours to limit further estimated losses."
    )
    print('=== Decision Explanation (rule-based fallback) ===')
    print(fallback)

## 10. Summary

Full pipeline results across all agents.

In [ ]:
print('=' * 60)
print('RE:GEN AI — Agent Pipeline Summary')
print('All data is simulated for capstone demonstration.')
print('=' * 60)
print()
print(f"Water:   {water_result['severity'].upper()} — "
      f"{water_result['total_wasted_liters']} L wasted | "
      f"Est. ₹{water_result['estimated_cost_inr']} | "
      f"{water_result['co2_equivalent_kg']} kg CO2")
print()
print(f"Energy:  {energy_result['severity'].upper()} — "
      f"{energy_result['total_wasted_kwh']} kWh wasted | "
      f"Est. ₹{energy_result['estimated_cost_inr']} | "
      f"{energy_result['co2_equivalent_kg']} kg CO2")
print()
print(f"Impact:  {impact_result['total_co2_saved_kg']} kg CO2 total | "
      f"{impact_result['trees_equivalent']} tree-equiv | "
      f"Rating: {impact_result['sustainability_rating']}")
print()
print(f"Decision: Top priority = {decision_result['top_priority_domain']} | "
      f"Total est. saving = ₹{decision_result['total_potential_saving_inr']}")
print()
print(f"RE:GEN Score: {score_result['before_score']}/100 (Grade {score_result['current_grade']}) "
      f"→ {score_result['after_score']}/100 (Grade {score_result['target_grade']}) "
      f"[+{score_result['improvement']} pts projected]")
print()
print('Disclaimer: RE:GEN AI is a prototype decision-support system.')
print('Not professional regulatory, financial, or engineering advice.')